<a href="https://colab.research.google.com/github/SaiSanthosh1508/Foundation-Models-From-Scratch/blob/main/Vision_Transformer_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
import torch
from torch import nn

# Architectural Components

## 1. Patch Embeddings

In [ ]:

class PatchEmbeddings(nn.Module):

  def __init__(self, img_sz : int, D : int):
    super().__init__()
    self.patch_size = 16
    self.img_sz = img_sz
    assert (img_sz % self.patch_size == 0), "image size must be divisible by patch size (16)"

    self.num_patches = (img_sz // self.patch_size)**2

    self.proj = nn.Conv2d(
        3,
        D,
        kernel_size = self.patch_size,
        stride = self.patch_size
    )

  def forward(self, x):
    # (B,C,H,W) --> (B,D,H',W') i.e H' = H/P & W' = W/P
    x = self.proj(x)

    # (B,D,H/P,W/P) --> (B,D,N) i.e N = H' * W'
    x = x.flatten(-2,-1)

    # (B,D,N) --> (B,N,D)
    x = x.transpose(-1,-2)

    return x

## 2. Vision Transformer Input Layer

In [ ]:
class ViTInputLayer(nn.Module):

  def __init__(self, img_sz : int, patch_size : int = 16, D : int = 768):
    super().__init__()

    self.embeddings = PatchEmbeddings(
        img_sz,D
    )
    num_patches = self.embeddings.num_patches
    # define the [CLS] token of dimension (1,D) to add to the image patches
    self.cls = nn.Parameter(torch.zeros(1,1, D))

    # create a learnable positional encoding
    self.pe = nn.Parameter(torch.zeros(1,num_patches+1,D))

  def forward(self, x):

    # x output shape: (B, N, D)
    x = self.embeddings(x)
    B, N, D = x.shape

    # Expand the cls token to the batch size of the input
    # shape: (B, 1, D)
    cls_token = self.cls.expand(B, -1, -1)

    # Concatenate along the N (patch) dimension
    # (B, 1, D) + (B, N, D) -> (B, N+1, D)
    x = torch.cat([cls_token, x], dim=1)

    # Add the learnable pe to the x
    # (B, N+1, D) + (1, N+1, D) -> (B, N+1, D)
    x = x + self.pe

    return x

## 3. Multi Head Attention

In [ ]:
import math

class MultiHeadAttention(nn.Module):

  def __init__(self, D : int, num_heads : int):
    super().__init__()
    self.D = D
    self.num_heads = num_heads

    assert (D % num_heads == 0), "D must be divisible by num_heads"
    self.d_head = D // num_heads
    self.W_q = nn.Linear(D, D)
    self.W_k = nn.Linear(D, D)
    self.W_v = nn.Linear(D, D)
    self.W_o = nn.Linear(D, D)

  def attention(self, q, k, v):

    d_head = q.shape[-1]
    scaled_attn = torch.matmul(q,k.transpose(-1,-2)) / math.sqrt(d_head)
    attn = torch.nn.functional.softmax(scaled_attn, dim=-1)

    return torch.matmul(attn, v)

  def forward(self, x):
    # (B,N+1,D) --> (B,N+1,D)
    B,N,_ = x.shape
    q = self.W_q(x)
    k = self.W_k(x)
    v = self.W_v(x)

    # (B,N+1,D) --> (B,N+1,num_heads,d_head) --> (B,num_heads,N+1,d_head)
    q = q.view(q.shape[0], q.shape[1], self.num_heads, self.d_head).transpose(1,2)
    k = k.view(k.shape[0], k.shape[1], self.num_heads, self.d_head).transpose(1,2)
    v = v.view(v.shape[0], v.shape[1], self.num_heads, self.d_head).transpose(1,2)

    # (B,num_heads,N+1,N+1) --> (B,num_heads,N+1,d_head)
    output = self.attention(q,k,v)

    output = output.transpose(1, 2).contiguous()

    # (B, N, H, d_h) -> (B, N, D)
    output = output.view(B, N, self.D)
    # (B,N+1,D) --> (B,N+1,D)
    return self.W_o(output)

## 4. MLP Block

In [ ]:
class MLP(nn.Module):

  def __init__(self, D : int):
    super().__init__()
    self.linear1 = nn.Linear(D, D*4)
    self.gelu = nn.GELU()
    self.linear2 = nn.Linear(D*4,D)

  def forward(self, x):

    return self.linear2(self.gelu(self.linear1(x)))

## 5. Encoder Block

In [ ]:
class EncoderBlock(nn.Module):

  def __init__(self, D : int, num_heads : int):
    super().__init__()
    self.D = D
    self.num_heads = num_heads

    self.norm1 = nn.LayerNorm(D)
    self.norm2 = nn.LayerNorm(D)

    self.attention = MultiHeadAttention(D, num_heads)
    self.mlp = MLP(D)

  def forward(self, x):

    x_norm1 = self.norm1(x)
    attn_out = self.attention(x_norm1)

    x = attn_out + x

    x_norm2 = self.norm2(x)
    mlp_out = self.mlp(x_norm2)

    return x + mlp_out

## Encoder

In [ ]:
class Encoder(nn.Module):

  def __init__(self, D : int, num_heads : int, N : int = 6):
    super().__init__()

    self.encoders = nn.ModuleList([
        EncoderBlock(D,num_heads) for _ in range(N)
    ])

  def forward(self, x):

    for layer in self.encoders:
      x = layer(x)

    return x

# Vision Transformer

In [ ]:
class ViT(nn.Module):

  def __init__(self, img_sz : int, num_class : int,patch_size : int = 16, D : int = 768, num_heads : int = 8, N : int = 6):
    super().__init__()

    self.input_layer = ViTInputLayer(img_sz,patch_size,D)
    self.encoder = Encoder(D,num_heads,N)
    self.norm = nn.LayerNorm(D)
    self.mlp_head = nn.Linear(D, num_class)

  def forward(self, x):

    x = self.input_layer(x)
    x = self.encoder(x)

    x = x[:,0]
    logits = self.mlp_head(self.norm(x))
    return logits


# Model Training

In [ ]:
!pip install -q transformers datasets

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Data preprocessing

In [ ]:
from datasets import load_dataset

ds = load_dataset("jonathan-roberts1/RSI-CB256",split="train")
ds

In [ ]:
ds

In [ ]:
ds = ds.remove_columns("label_2")
ds

In [ ]:
ds = ds.train_test_split(test_size=0.25)
ds

In [ ]:
ds = ds['test'].train_test_split(test_size=0.3)
ds

In [ ]:
classes = ds['train'].features['label_1'].names
classes

In [ ]:
from torchvision import transforms

IMG_SZ = 224
data_transforms = transforms.Compose([
      transforms.Resize((IMG_SZ, IMG_SZ)),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
  ])
def preprocess_fn(batch):

  batch['image'] = [data_transforms(img.convert('RGB')) for img in batch['image']]
  return batch

In [ ]:
processed_ds = ds.map(preprocess_fn,batched=True)
processed_ds

In [ ]:
processed_ds.set_format(type="torch")

In [ ]:
from torch.utils.data import DataLoader

train_dl = DataLoader(processed_ds['train'],batch_size=32,shuffle=True)
val_dl = DataLoader(processed_ds['test'],batch_size=32,shuffle=False)

In [ ]:
len(train_dl), len(val_dl)

## Model Training

In [ ]:
model = ViT(img_sz = IMG_SZ,num_class = len(classes), patch_size = 16, D = 768, num_heads = 8, N = 6)
model

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

from torch.optim import Adam

optimizer = Adam(model.parameters(),lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
for batch in train_dl:
  print(batch['label_1'].shape)
  break

In [ ]:
from tqdm import tqdm

def train(model, train_dl, val_dl, epochs, optimizer, loss_fn, device ):

  for epoch in range(epochs):

    model.train()
    train_loss = 0.0

    train_loop = tqdm(train_dl, desc=f"[Training]")
    for batch in train_loop:

      images = batch['image'].to(device)
      labels = batch['label_1'].to(device)

      logits = model(images)

      loss = loss_fn(logits,labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      train_loss += loss.item()
      train_loop.set_postfix_str(f"loss: {loss.item():.4f}")

    avg_train_loss = train_loss / len(train_dl)


    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():

      val_loop = tqdm(val_dl, desc=f"[Validation]")
      for batch in val_loop:

        images = batch['image'].to(device)
        labels = batch['label_1'].to(device)

        logits = model(images)

        loss = loss_fn(logits,labels)

        val_loss += loss.item()

        preds = torch.argmax(logits, dim = -1)

        total += labels.shape[0]
        correct += (preds == labels).sum().item()

        val_loop.set_postfix_str(f"loss: {loss.item():.4f}")
      avg_val_loss = val_loss / len(val_dl)
      accuracy = 100 * correct / total
      print(f"Epoch {epoch+1} Test Loss: {avg_val_loss:.4f} | Accuracy: {accuracy:.2f}%")

  print("Training finished")

In [ ]:
train(model,train_dl,val_dl,5,optimizer,loss_fn,device)

## Inference

In [ ]:
import matplotlib.pyplot as plt

def preprocess_image(img):

  original_img = img.convert('RGB')
  processed_img = data_transforms(original_img)
  return original_img, processed_img

def inference(model, image, classes, device):

  model.eval()
  with torch.no_grad():

    # We now get both the original image for display and the processed for the model
    original_img, processed_img = preprocess_image(image)

    # The processed image for the model needs unsqueezing and moving to device
    processed_img = processed_img.unsqueeze(0).to(device)
    logits = model(processed_img)
    class_idx = torch.argmax(logits,dim=-1)

    fig = plt.figure()
    ax = fig.add_subplot(1,1,1)

    ax.imshow(original_img)
    ax.axis('off')
    class_name = classes[class_idx]
    ax.set_title(f"Predicted class: {class_name}")


In [ ]:
inference(model, ds['test'][108]['image'], classes, device)